In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from IPython.display import clear_output
!pip install kaggle
import os
os.environ['KAGGLE_USERNAME'] = 'ramindusulakkana'
os.environ['KAGGLE_KEY'] = '28949047'
clear_output()

In [3]:
!kaggle datasets download -d "vipoooool/new-plant-diseases-dataset" -p /content/drive/MyDrive/datasets/

Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
new-plant-diseases-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
import zipfile

zip_path = '/content/drive/MyDrive/datasets/new-plant-diseases-dataset.zip'
extract_path = '/content/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [5]:
import tensorflow as tf
import os
data_dir = '/content/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'

class_names = [d for d in os.listdir(data_dir)
               if os.path.isdir(os.path.join(data_dir, d)) and 'healthy' not in d.lower()]

training_set = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="categorical",
    class_names=class_names,
    color_mode="rgb",
    batch_size=32,
    image_size=(224, 224),
    shuffle=True
)

Found 48001 files belonging to 26 classes.


In [6]:
data_dir = '/content/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid'

class_names = [d for d in os.listdir(data_dir)
               if os.path.isdir(os.path.join(data_dir, d)) and 'healthy' not in d.lower()]

validation_set = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="categorical",
    class_names=class_names,
    color_mode="rgb",
    batch_size=32,
    image_size=(224, 224),
    shuffle=True
)

Found 12000 files belonging to 26 classes.


In [7]:
print("Classes used for training:")
print(class_names)

Classes used for training:
['Grape___Black_rot', 'Corn_(maize)___Common_rust_', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Bacterial_spot', 'Tomato___Leaf_Mold', 'Apple___Cedar_apple_rust', 'Apple___Black_rot', 'Cherry_(including_sour)___Powdery_mildew', 'Potato___Early_blight', 'Potato___Late_blight', 'Peach___Bacterial_spot', 'Grape___Esca_(Black_Measles)', 'Strawberry___Leaf_scorch', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Apple___Apple_scab', 'Tomato___Late_blight', 'Tomato___Septoria_leaf_spot', 'Tomato___Tomato_mosaic_virus', 'Orange___Haunglongbing_(Citrus_greening)', 'Squash___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Tomato___Early_blight', 'Tomato___Target_Spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Corn_(maize)___Northern_Leaf_Blight', 'Pepper,_bell___Bacterial_spot']


In [8]:
from tensorflow import keras
from tensorflow.keras.layers import Rescaling
training_set = training_set.map(lambda x, y: (Rescaling(1./255)(x), y))
validation_set = validation_set.map(lambda x, y: (Rescaling(1./255)(x), y))

In [9]:
from tensorflow.keras import layers
geometric_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
])

def apply_geometric_augmentation(image, label):
    image = geometric_augmentation(image)
    return image, label

training_set = training_set.map(apply_geometric_augmentation)

In [10]:
from tensorflow.keras.layers import Dense,Conv2D,MaxPool2D,Flatten,Dropout,BatchNormalization
from tensorflow.keras.models import Sequential
from keras.optimizers import AdamW, SGD

In [11]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.3)(x)
output = Dense(26, activation='softmax')(x)
model = Model(inputs=base.input, outputs=output)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(training_set, validation_data=validation_set, epochs=10)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/10
1501/1501 ━━━━━━━━━━━━━━━━━━━━ 863s 563ms/step - accuracy: 0.6615 - loss: 1.1521 - val_accuracy: 0.8367 - val_loss: 0.5020
Epoch 2/10
1501/1501 ━━━━━━━━━━━━━━━━━━━━ 826s 550ms/step - accuracy: 0.8544 - loss: 0.4433 - val_accuracy: 0.8633 - val_loss: 0.4077
Epoch 3/10
1501/1501 ━━━━━━━━━━━━━━━━━━━━ 831s 554ms/step - accuracy: 0.8627 - loss: 0.4056 - val_accuracy: 0.8841 - val_loss: 0.3422
Epoch 4/10
1501/1501 ━━━━━━━━━━━━━━━━━━━━ 834s 556ms/step - accuracy: 0.8711 - loss: 0.3833 - val_accuracy: 0.8723 - val_loss: 0.3754
Epoch 5/10
1501/1501 ━━━━━━━━━━━━━━━━━━━━ 825s 550ms/step - accuracy: 0.8723 - loss: 0.3827 - val_accuracy: 0.8755 - val_loss: 0.3644
Epoch 6/10
1501/1501 ━━━━━━━━━━━━━━━━━━━━ 826s 550ms/step - accuracy: 0.8780 - loss: 0.3708 - val_accuracy: 0.8924 - val_loss: 0.3168
Epoch 7/10
1501/1501 ━━━━━━━━━━━━━━━━━━━━ 818s 545ms/step - accuracy: 0.8780 - loss: 0.3639 - val_accuracy: 0.8963 - val_loss: 0.3155
Epoch 8/10
15

In [12]:
train_loss,train_acc = model.evaluate(training_set)

1501/1501 ━━━━━━━━━━━━━━━━━━━━ 796s 530ms/step - accuracy: 0.9170 - loss: 0.2442


In [13]:
val_loss,val_acc = model.evaluate(validation_set)

375/375 ━━━━━━━━━━━━━━━━━━━━ 15s 39ms/step - accuracy: 0.8951 - loss: 0.2985


In [14]:
model.save("/content/drive/MyDrive/Models/plant_model.keras")

In [15]:
model = tf.keras.models.load_model("/content/drive/MyDrive/Models/plant_model.keras")

In [24]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = '/content/test/test/AppleScab1.JPG'

img = image.load_img(img_path, target_size=(224, 224))

img_array = image.img_to_array(img)


img_array = np.expand_dims(img_array, axis=0)

preprocessed_image = img_array / 255.0

In [25]:
prediction = model.predict(preprocessed_image)
prediction,prediction.shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


(array([[1.80077123e-07, 4.59384131e-09, 4.52880595e-05, 2.31761974e-03,
         1.10486028e-04, 1.34117808e-02, 2.06134628e-05, 6.54231384e-03,
         6.60535670e-06, 1.57177635e-03, 2.34853011e-03, 6.80345158e-09,
         8.49217372e-07, 6.89040231e-12, 9.72733438e-01, 3.88039247e-04,
         5.46237061e-05, 1.06789230e-06, 2.95315772e-06, 1.82170362e-10,
         3.34303891e-08, 2.01228337e-04, 1.22508747e-04, 2.46838522e-06,
         2.04207922e-07, 1.17723415e-04]], dtype=float32),
 (1, 26))

In [26]:
result_index = np.argmax(prediction)
result_index

np.int64(14)

In [27]:
class_names[result_index]

'Apple___Apple_scab'

In [28]:
# 1) Save a copy of the model in /content (same place as app.py later)
model.save("plant_model.keras")

# 2) Save your class_names list to a json file
import json

with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print("Saved plant_model.keras and class_names.json in /content")


Saved plant_model.keras and class_names.json in /content


In [29]:
%%writefile app.py
import streamlit as st
from PIL import Image
import numpy as np
import tensorflow as tf
import json

# ---- Load model and class names ----
@st.cache_resource
def load_model():
    model = tf.keras.models.load_model("plant_model.keras")
    return model

@st.cache_resource
def load_class_names():
    with open("class_names.json", "r") as f:
        class_names = json.load(f)
    return class_names

model = load_model()
CLASS_NAMES = load_class_names()

IMG_SIZE = (224, 224)  # you trained with (224, 224, 3)

# ---- Streamlit UI ----
st.title("🌿 Plant Disease Detection")
st.write("Upload a plant leaf image and the model will predict the disease.")

uploaded_file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # show image
    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded Image", use_column_width=True)

    # preprocess exactly like training: resize + /255.0
    img = image.resize(IMG_SIZE)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # predict
    preds = model.predict(img_array)
    pred_index = np.argmax(preds[0])
    pred_class = CLASS_NAMES[pred_index]
    confidence = float(np.max(preds[0]) * 100)

    st.success(f"🌱 **Predicted Disease:** {pred_class}")
    st.write(f"🔍 Confidence: `{confidence:.2f}%`")


Writing app.py


# New Section